# Preoperative HbA1c CDS Reproducible Workflow

This notebook is the shareable procedure for the pre-surgical HbA1c risk workflow. It is organized so a reviewer can run it without institutional credentials using the sample data path, then switch to Cosmos-backed data by providing environment variables from `.env.example`.

Default behavior uses sample data and generates reviewable FHIR resources without sending alerts.

## Author & Acknowledgments

Corresponding author: David Balkcom, MPH, Principal Data Engineer, Quality Measurement Group, IT Vendor Services, United States. Contact: castle.palaces8l@icloud.com.

Acknowledgments: Andrea Pitkus, PhD.

## HbA1c Specimen Clarification

A serum HbA1c test method does not exist. HbA1c measures the glucose attached to hemoglobin, which is a protein found strictly inside red blood cells. Serum is the liquid portion of the blood after the red blood cells and clotting factors have been removed, meaning it contains no hemoglobin to test. For HbA1c testing, a whole blood sample (which includes red blood cells) is always required. Laboratories process this using methods like HPLC (High-Performance Liquid Chromatography) or immunoassays.

## Example External Lab Value Flow

The flow below summarizes how external HbA1c values move from EHI or another clinical source into the preoperative CDS workflow.

```mermaid
flowchart TD
    A["EHI or External Clinical Source"] --> B["FHIR Observation<br/>HbA1c LOINC 4548-4"]
    B --> C["Azure Cosmos DB<br/>Clinical data layer"]
    C --> D["Observation.valueQuantity %<br/>Observation.effectiveDateTime"]
    D --> E["Pre-op HbA1c CDS Pipeline"]
    E --> F["Risk Tiering<br/>info / warning / critical"]
    F --> G["FHIR-ready CDS Resources<br/>CommunicationRequest / Task"]
    G --> H["Epic workflow integration review"]
```

## Procedure

1. Load the pipeline code and runtime configuration.
2. Select the data source: sample data for replication or Cosmos for local execution.
3. Validate the cohort, HbA1c lab, and vitals fields needed by the workflow.
4. Merge patient, lab, and vitals records into one analytic table.
5. Optionally estimate missing HbA1c values if a training set and `adelie` are available.
6. Apply HbA1c risk tiers: Low `<7.5`, Medium `7.5-8.5`, High `>8.5` by default.
7. Map tiers to CDS workflow actions and FHIR artifact previews.
8. Save run metadata and compact outputs for review.

## Environment Setup

In a fresh Python environment, install the base runtime dependencies before executing the notebook:

```bash
python3 -m venv .venv
.venv/bin/pip install python-dotenv pandas numpy requests matplotlib azure-cosmos
```

`adelie` is optional and only needed when `RUN_PREDICTION=true`. On macOS it requires OpenMP first, for example `libomp` via Homebrew or `llvm-openmp` via conda.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in candidate_roots if (path / "cds-epic-cosmos.py").exists())

if load_dotenv is not None:
    load_dotenv(REPO_ROOT / ".env")

pipeline_path = REPO_ROOT / "cds-epic-cosmos.py"
spec = importlib.util.spec_from_file_location("preop_hba1c_pipeline", pipeline_path)
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

print(f"Repo root: {REPO_ROOT}")
print(f"Pipeline source: {pipeline_path.name}")

## Parameters

Use `DATA_SOURCE=sample` for a portable run. Use `DATA_SOURCE=cosmos` after setting the Cosmos variables in `.env` or the execution environment. The notebook keeps `DRY_RUN=True` so FHIR resources are generated for review only.

In [ ]:
DATA_SOURCE = os.getenv("DATA_SOURCE", "sample").strip().lower()
PREOP_DATE = os.getenv("PREOP_DATE", "2026-05-01").strip()

LOW_THRESHOLD = float(os.getenv("HBA1C_LOW_THRESHOLD", "7.5"))
HIGH_THRESHOLD = float(os.getenv("HBA1C_HIGH_THRESHOLD", "8.5"))

RUN_PREDICTION = os.getenv("RUN_PREDICTION", "false").strip().lower() in {"1", "true", "yes"}
TRAINING_CSV = os.getenv("TRAINING_CSV") or None

DRY_RUN = True

config = {
    "data_source": DATA_SOURCE,
    "preop_date": PREOP_DATE,
    "low_threshold": LOW_THRESHOLD,
    "high_threshold": HIGH_THRESHOLD,
    "run_prediction": RUN_PREDICTION,
    "training_csv": TRAINING_CSV,
    "dry_run": DRY_RUN,
}
config

## Load Data

The sample data mirrors the expected shape of the Cosmos query outputs and covers each risk tier plus an unknown HbA1c case. This makes the notebook executable by outside reviewers without sending sample patient data.

In [ ]:
def require_env(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def load_sample_data(preop_date: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cohort = pd.DataFrame([
        {
            "patient_id": "example-001",
            "encounter_id": "enc-1001",
            "preop_date": preop_date,
            "birthdate": "1971-04-09",
            "gender": "female",
            "surgical_risk": "moderate",
        },
        {
            "patient_id": "example-002",
            "encounter_id": "enc-1002",
            "preop_date": preop_date,
            "birthdate": "1964-11-21",
            "gender": "male",
            "surgical_risk": "high",
        },
        {
            "patient_id": "example-003",
            "encounter_id": "enc-1003",
            "preop_date": preop_date,
            "birthdate": "1983-02-15",
            "gender": "female",
            "surgical_risk": "low",
        },
        {
            "patient_id": "example-004",
            "encounter_id": "enc-1004",
            "preop_date": preop_date,
            "birthdate": "1958-07-30",
            "gender": "unknown",
            "surgical_risk": "moderate",
        },
    ])

    labs = pd.DataFrame([
        {"patient_id": "example-001", "a1c": 6.9, "lab_time": "2026-04-18T09:30:00Z"},
        {"patient_id": "example-002", "a1c": 8.1, "lab_time": "2026-04-17T13:10:00Z"},
        {"patient_id": "example-003", "a1c": 9.2, "lab_time": "2026-04-16T08:05:00Z"},
    ])

    vitals = pd.DataFrame([
        {"patient_id": "example-001", "sbp": 126, "dbp": 78, "bmi": 29.3, "vitals_time": "2026-05-01T06:10:00Z"},
        {"patient_id": "example-002", "sbp": 142, "dbp": 86, "bmi": 34.1, "vitals_time": "2026-05-01T06:15:00Z"},
        {"patient_id": "example-003", "sbp": 138, "dbp": 84, "bmi": 32.7, "vitals_time": "2026-05-01T06:20:00Z"},
        {"patient_id": "example-004", "sbp": 130, "dbp": 82, "bmi": 31.0, "vitals_time": "2026-05-01T06:25:00Z"},
    ])

    return cohort, labs, vitals


if DATA_SOURCE == "cosmos":
    settings = pipeline.Settings(
        cosmos_url=require_env("COSMOS_URL"),
        cosmos_key=require_env("COSMOS_KEY"),
        database_id=require_env("DATABASE_ID"),
        container_patient=os.getenv("CONTAINER_PATIENT", "PATIENT").strip(),
        container_labs=os.getenv("CONTAINER_LABS", "NSQIP_PREOP_LABS").strip(),
        container_vitals=os.getenv("CONTAINER_VITALS", "PREOP_VITALS").strip(),
        fhir_base=os.getenv("FHIR_BASE", "").strip(),
        oauth_bearer=os.getenv("OAUTH_BEARER", "").strip(),
        preop_date=PREOP_DATE,
        low_threshold=LOW_THRESHOLD,
        high_threshold=HIGH_THRESHOLD,
        H=None,
        delta=None,
        dry_run=DRY_RUN,
        plot=False,
        training_csv=TRAINING_CSV,
        enable_prediction=RUN_PREDICTION,
    )
    client = pipeline.cosmos_client(settings)
    db = client.get_database_client(settings.database_id)
    df_pat = pipeline.load_preop_cohort(db, settings)
    df_labs = pipeline.load_a1c_labs(db, settings)
    df_vitals = pipeline.load_vitals(db, settings)
else:
    settings = None
    df_pat, df_labs, df_vitals = load_sample_data(PREOP_DATE)

print({"patients": len(df_pat), "labs": len(df_labs), "vitals": len(df_vitals)})

## Validate Inputs

These checks make the notebook fail early if an exported or queried dataset is missing fields needed by downstream logic.

In [ ]:
required_columns = {
    "patient": {"patient_id", "encounter_id", "preop_date", "birthdate", "gender", "surgical_risk"},
    "labs": {"patient_id", "a1c", "lab_time"},
    "vitals": {"patient_id", "sbp", "dbp", "bmi", "vitals_time"},
}

frames = {"patient": df_pat, "labs": df_labs, "vitals": df_vitals}
for name, required in required_columns.items():
    missing = sorted(required.difference(frames[name].columns))
    if missing:
        raise ValueError(f"{name} data is missing columns: {missing}")

if df_pat.empty:
    raise ValueError(f"No pre-op cohort found for PREOP_DATE={PREOP_DATE}")

print("Input validation passed")

## Build Analytic Table

Patient records define the cohort. Lab and vitals records are left-joined so missing HbA1c remains visible and can be handled explicitly.

In [ ]:
analysis_df = df_pat.merge(df_labs, on="patient_id", how="left").merge(df_vitals, on="patient_id", how="left")
analysis_df = pipeline.add_age(analysis_df)

analysis_df["a1c"] = pd.to_numeric(analysis_df["a1c"], errors="coerce")
analysis_df["lab_time"] = pd.to_datetime(analysis_df["lab_time"], errors="coerce", utc=True)
analysis_df["vitals_time"] = pd.to_datetime(analysis_df["vitals_time"], errors="coerce", utc=True)

analysis_df

## Optional Missing HbA1c Estimation

Prediction is disabled by default. Enable it only when the execution environment has `adelie` and either enough in-scope measured HbA1c rows or a `TRAINING_CSV` with compatible columns.

In [ ]:
model_info = {"trained": False, "reason": "prediction_disabled"}

if RUN_PREDICTION:
    pred_a1c, model_info = pipeline.fit_predict_a1c_with_adelie(analysis_df, TRAINING_CSV)
    analysis_df["a1c_predicted"] = pred_a1c
    analysis_df["a1c_used"] = analysis_df["a1c"].where(analysis_df["a1c"].notna(), analysis_df["a1c_predicted"])
else:
    analysis_df["a1c_predicted"] = np.nan
    analysis_df["a1c_used"] = analysis_df["a1c"]

model_info

## Apply Risk Tiers and CDS Actions

Risk tiers are computed from measured HbA1c when present, otherwise from the optional predicted value. Unknown HbA1c values stay unknown rather than being silently treated as low risk.

In [ ]:
tiers = analysis_df["a1c_used"].apply(
    lambda value: pipeline.tier_from_a1c(value, LOW_THRESHOLD, HIGH_THRESHOLD)
)

analysis_df["tier"] = tiers.apply(lambda item: item[0])
analysis_df["risk_category"] = tiers.apply(lambda item: item[1])
analysis_df["tier_metrics"] = tiers.apply(lambda item: item[2])
analysis_df["workflow_action"] = analysis_df["tier"].apply(pipeline.workflow_action_from_tier)

analysis_df["cds_message"] = analysis_df.apply(
    lambda row: pipeline.build_comm_payload(
        float(row["tier_metrics"]["a1c_used"])
        if not np.isnan(row["tier_metrics"]["a1c_used"])
        else np.nan,
        row["tier"],
        row["tier_metrics"],
    ),
    axis=1,
)

summary_cols = [
    "patient_id",
    "encounter_id",
    "preop_date",
    "a1c",
    "a1c_predicted",
    "a1c_used",
    "risk_category",
    "workflow_action",
    "cds_message",
]
analysis_df[summary_cols]

## Cohort Summary

The bar chart is intentionally simple because this output is meant for quick clinical review and run-to-run comparison.

In [ ]:
risk_order = ["Low", "Medium", "High", "Unknown"]
risk_counts = analysis_df["risk_category"].value_counts(dropna=False).reindex(risk_order, fill_value=0)

ax = risk_counts.plot(kind="bar", figsize=(7, 4), color=["#4c956c", "#f2c14e", "#c44536", "#7d8597"])
ax.set_title("Preoperative HbA1c Risk Category")
ax.set_xlabel("")
ax.set_ylabel("Patient Count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

risk_counts.to_frame(name="patient_count")

## FHIR Preview

This cell previews the FHIR resources that would be posted by the pipeline. It does not call the FHIR endpoint. Critical cases create both a `CommunicationRequest` and a `Task`; non-critical CDS actions create a `CommunicationRequest`.

In [ ]:
def preview_fhir_payloads(df: pd.DataFrame) -> list[dict]:
    payloads = []

    for _, row in df.iterrows():
        action = row.get("workflow_action", "None")
        if action == "None":
            continue

        patient_ref = f"Patient/{row['patient_id']}"
        encounter_ref = f"Encounter/{row['encounter_id']}"

        comm = {
            "resourceType": "CommunicationRequest",
            "status": "active",
            "subject": {"reference": patient_ref},
            "encounter": {"reference": encounter_ref},
            "payload": [{"contentString": row.get("cds_message", "Pre-op risk message")}],
            "reasonCode": [{"text": "Pre-op Glycemic Surgical Risk"}],
        }

        if action == "Block":
            comm["priority"] = "stat"
            comm["category"] = [{"text": "Stop-the-line / Clinical huddle"}]
        elif action == "Warn":
            comm["priority"] = "urgent"
            comm["category"] = [{"text": "Pre-op warning"}]
        elif action == "Info":
            comm["priority"] = "routine"

        payloads.append(comm)

        if action == "Block":
            payloads.append({
                "resourceType": "Task",
                "status": "requested",
                "intent": "order",
                "description": "High surgical diabetes risk - HITL sign-off required.",
                "for": {"reference": patient_ref},
                "encounter": {"reference": encounter_ref},
                "priority": "stat",
            })

    return payloads


fhir_payloads = preview_fhir_payloads(analysis_df)
print(json.dumps(fhir_payloads, indent=2))

## Run Metadata

Capture the parameters and package versions needed to compare runs across sites.

In [ ]:
run_metadata = {
    **config,
    "executed_on": date.today().isoformat(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": getattr(plt.matplotlib, "__version__", "unknown"),
    "pipeline_path": str(pipeline_path.relative_to(REPO_ROOT)),
    "n_patients": int(len(analysis_df)),
    "n_fhir_preview_resources": int(len(fhir_payloads)),
}

print(json.dumps(run_metadata, indent=2))